In [1]:
import os
import h5py
import numpy as np
import tensorflow as tf
from tensorflow import keras as K
from tensorflow.keras.layers import Input, Conv2D, Conv2DTranspose, Dense, Flatten, Reshape
from tensorflow.keras.layers import BatchNormalization, ReLU
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.preprocessing import StandardScaler
import pickle

np.random.seed(42)
tf.random.set_seed(42)

loadPath = '/home/sz4544/EEG-motor-imagery-main/project/'
os.makedirs(os.path.join(loadPath, "models"), exist_ok=True)

# ---------------------------
# load train / valid / test
# ---------------------------
f_train = h5py.File(os.path.join(loadPath, "train1800_raw_EEG.h5"), "r")
tr_data = f_train['data'][:]
f_train.close()

f_valid = h5py.File(os.path.join(loadPath, "valid360_raw_EEG.h5"), "r")
val_data = f_valid['data'][:]
f_valid.close()

f_test = h5py.File(os.path.join(loadPath, "test360_raw_EEG.h5"), "r")
ts_data = f_test['data'][:]
f_test.close()

print("train:", tr_data.shape)
print("valid:", val_data.shape)
print("test:", ts_data.shape)

# ---------------------------
# scaler fit on train only
# ---------------------------
xtr_flat = np.squeeze(tr_data).reshape((-1, 64))
xval_flat = np.squeeze(val_data).reshape((-1, 64))
xts_flat = np.squeeze(ts_data).reshape((-1, 64))

scaler = StandardScaler()
xtr_scaled = scaler.fit_transform(xtr_flat)
xval_scaled = scaler.transform(xval_flat)
xts_scaled = scaler.transform(xts_flat)

with open(os.path.join(loadPath, "models", "latent_autoencoder_scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

x_train = xtr_scaled.reshape((-1, 640, 64, 1)).astype(np.float32)
x_valid = xval_scaled.reshape((-1, 640, 64, 1)).astype(np.float32)
x_test = xts_scaled.reshape((-1, 640, 64, 1)).astype(np.float32)

print("x_train:", x_train.shape)
print("x_valid:", x_valid.shape)
print("x_test:", x_test.shape)

# ---------------------------
# autoencoder
# ---------------------------
latent_dim = 128

inputs = Input(shape=(640, 64, 1))

x = Conv2D(16, (9, 5), strides=(2, 2), padding='same')(inputs)   # 320 x 32
x = BatchNormalization()(x)
x = ReLU()(x)

x = Conv2D(32, (9, 5), strides=(2, 2), padding='same')(x)        # 160 x 16
x = BatchNormalization()(x)
x = ReLU()(x)

x = Conv2D(64, (9, 5), strides=(2, 2), padding='same')(x)        # 80 x 8
x = BatchNormalization()(x)
x = ReLU()(x)

x = Flatten()(x)
latent = Dense(latent_dim, name="latent_vector")(x)

# decoder
x = Dense(80 * 8 * 64)(latent)
x = ReLU()(x)
x = Reshape((80, 8, 64))(x)

x = Conv2DTranspose(32, (9, 5), strides=(2, 2), padding='same')(x)  # 160 x 16
x = BatchNormalization()(x)
x = ReLU()(x)

x = Conv2DTranspose(16, (9, 5), strides=(2, 2), padding='same')(x)  # 320 x 32
x = BatchNormalization()(x)
x = ReLU()(x)

x = Conv2DTranspose(8, (9, 5), strides=(2, 2), padding='same')(x)   # 640 x 64
x = BatchNormalization()(x)
x = ReLU()(x)

outputs = Conv2D(1, (3, 3), padding='same', activation='linear')(x)

autoencoder = Model(inputs, outputs, name="eeg_autoencoder")
encoder = Model(inputs, latent, name="eeg_encoder")

autoencoder.compile(
    optimizer=K.optimizers.Adam(learning_rate=1e-3),
    loss='mse'
)

print(autoencoder.summary())
print(encoder.summary())

ae_path = os.path.join(loadPath, "models", "latent_autoencoder.keras")
enc_path = os.path.join(loadPath, "models", "latent_encoder.keras")

es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1)
mc = ModelCheckpoint(ae_path, monitor='val_loss', save_best_only=True, verbose=1)

history = autoencoder.fit(
    x_train, x_train,
    validation_data=(x_valid, x_valid),
    epochs=200,
    batch_size=32,
    shuffle=True,
    callbacks=[es, mc],
    verbose=1
)

# save final models
autoencoder.save(ae_path)
encoder.save(enc_path)

print("saved autoencoder:", ae_path)
print("saved encoder:", enc_path)

I0000 00:00:1774223130.575384  114470 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


train: (1800, 640, 64)
valid: (360, 640, 64)
test: (360, 640, 64)
x_train: (1800, 640, 64, 1)
x_valid: (360, 640, 64, 1)
x_test: (360, 640, 64, 1)


I0000 00:00:1774223133.381011  114470 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22272 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1774223133.381584  114470 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22253 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9


Model: "eeg_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 640, 64, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 320, 32, 16)    │           736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 320, 32, 16)    │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 320, 32, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 160, 16, 32)    │        23,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 160, 16, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 160, 16, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 80, 8, 64)      │        92,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 80, 8, 64)      │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 80, 8, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 40960)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent_vector (Dense)           │ (None, 128)            │     5,243,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 40960)          │     5,283,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 40960)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 80, 8, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose                │ (None, 160, 16, 32)    │        92,192 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 160, 16, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 160, 16, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_1              │ (None, 320, 32, 16)    │        23,056 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 320, 32, 16)    │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 320, 32, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

...

22/57 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.1763

I0000 00:00:1774223144.238035  114703 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


56/57 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.0747

I0000 00:00:1774223145.059412  114702 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_5806__.46


57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - loss: 1.0731
Epoch 1: val_loss improved from None to 4.56405, saving model to /home/sz4544/EEG-motor-imagery-main/project/models/latent_autoencoder.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 18s 157ms/step - loss: 0.9805 - val_loss: 4.5640
Epoch 2/200
56/57 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.8547
Epoch 2: val_loss improved from 4.56405 to 1.00284, saving model to /home/sz4544/EEG-motor-imagery-main/project/models/latent_autoencoder.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.8290 - val_loss: 1.0028
Epoch 3/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.7562
Epoch 3: val_loss did not improve from 1.00284
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.7364 - val_loss: 1.4584
Epoch 4/200
55/57 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6884
Epoch 4: val_loss improved from 1.00284 to 0.87250, saving model to /home/sz4544/EEG-motor-imagery-main/project/models/latent_autoencoder.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss